# The 2026 Travel Burden Index

Three host countries, sixteen venues, a continent's worth of distance. This
notebook builds a defensible, from-scratch **burden index** for every team in
the 2026 World Cup group stage — total kilometres flown, time-zones crossed,
altitude breathed, and rest denied — then asks the only honest follow-up:
*does the ranking survive if you reweight the ingredients?*

Data is static (the final-draw schedule + venue coordinates), so there's no
scraping and nothing to go stale. All logic lives in `mlfootball/travel.py`;
this notebook narrates it and writes `site/data/travel.json` for the web app.

In [1]:
import json
from pathlib import Path

from mlfootball import travel as T

print(f"{len(T.VENUES)} venues · {len(T.SCHEDULE)} group-stage matches · "
      f"{sum(len(v) for v in T.GROUPS.values())} teams")

16 venues · 72 group-stage matches · 48 teams


## 1 · One team's journey

Start concrete. Czechia drew the cruellest itinerary in the field: a Mexican
opener at altitude in Guadalajara, a transcontinental hop to Atlanta, then back
up to 2,200 m in Mexico City.

In [2]:
cze = T.team_travel("Czechia", "A")
for leg in cze.legs:
    tag = f"  ✈ {leg['leg_km']} km" if leg["leg_km"] else ""
    alt = f"  ({leg['alt']} m)" if leg["alt"] >= T.ALT_THRESHOLD else ""
    print(f"{leg['date']}  {leg['city']:14}{alt:10} vs {leg['opponent']}{tag}")
print(f"\nTotal {cze.total_km} km · {cze.tz_changes} tz-changes · "
      f"{cze.alt_matches} altitude matches · shortest rest {cze.min_rest} d")

2026-06-11  Guadalajara     (1560 m) vs South Korea
2026-06-18  Atlanta                  vs South Africa  ✈ 2373 km
2026-06-24  Mexico City     (2200 m) vs Mexico  ✈ 2171 km

Total 4544 km · 4 tz-changes · 2 altitude matches · shortest rest 6 d


## 2 · The index

Four sub-burdens, each **z-scored across the 48 teams** so they're comparable,
then combined as a weighted sum. Recovery enters negatively (less rest = more
burden). Default weights: distance .35, timezone .25, altitude .25, recovery .15.

In [3]:
table = T.burden_table()
print(f"{'#':>2}  {'Team':14} {'Grp':3} {'km':>5} {'tz':>2} {'alt':>5} {'rest':>4}  burden")
for r in table[:10]:
    print(f"{r['rank']:>2}  {r['team']:14} {r['group']:3} {r['total_km']:>5} "
          f"{r['tz_changes']:>2} {int(r['altitude_load']):>5} {r['min_rest']:>4}  {r['burden']:+.3f}")
print("...")
for r in table[-3:]:
    print(f"{r['rank']:>2}  {r['team']:14} {r['group']:3} {r['total_km']:>5} "
          f"{r['tz_changes']:>2} {int(r['altitude_load']):>5} {r['min_rest']:>4}  {r['burden']:+.3f}")

 #  Team           Grp    km tz   alt rest  burden
 1  Czechia        A    4544  4  3760    6  +1.740
 2  Colombia       K    2915  2  3760    4  +1.306
 3  Algeria        J    4798  4     0    5  +1.305
 4  DR Congo       K    3660  3  1560    4  +1.300
 5  South Africa   A    3943  4  2200    6  +1.251
 6  Bosnia & Herzegovina B    5058  3     0    6  +0.947
 7  Uzbekistan     K    2349  2  2200    4  +0.828
 8  Uruguay        H    2439  2  1560    5  +0.496
 9  Ecuador        E    3405  2     0    5  +0.478
10  Spain          H    2373  2  1560    5  +0.476
...
46  Argentina      J     739  0     0    5  -0.732
47  Paraguay       D     505  0     0    5  -0.803
48  Egypt          G     391  0     0    5  -0.837


## 3 · Who got screwed by the draw?

Average the burden over each group's four teams.

In [4]:
for r in T.group_burden():
    bar = "█" * int(max(0, r["mean_burden"]) * 20)
    print(f"Group {r['group']}  {r['mean_burden']:+.2f}  {bar}")

Group K  +0.85  ████████████████
Group A  +0.72  ██████████████
Group J  +0.27  █████
Group H  +0.21  ████
Group B  +0.15  ███
Group E  -0.01  
Group L  -0.10  
Group F  -0.20  
Group G  -0.43  
Group C  -0.43  
Group D  -0.50  
Group I  -0.53  


## 4 · Sensitivity — is the ranking real or an artefact of the weights?

Re-run the index under five very different weightings and measure Kendall's τ
against the default ordering. High τ = the ranking is a property of the
schedule, not of our weight choices.

In [5]:
sens = T.sensitivity()
print(f"mean τ vs default: {sens['mean_tau']}\n")
for sc in sens["scenarios"]:
    print(f"  {sc['scenario']:16} τ = {sc['tau']:+.3f}")
print("\nMost weight-robust teams (smallest rank spread across scenarios):")
for s in sorted(sens["stability"], key=lambda x: x["spread"])[:5]:
    print(f"  {s['team']:14} rank {s['best']}–{s['worst']}")

mean τ vs default: 0.801

  equal            τ = +0.787
  distance_heavy   τ = +0.865
  altitude_heavy   τ = +0.904
  timezone_heavy   τ = +0.906
  recovery_heavy   τ = +0.544

Most weight-robust teams (smallest rank spread across scenarios):
  Argentina      rank 44–46
  Paraguay       rank 45–47
  Egypt          rank 46–48
  DR Congo       rank 1–5
  Ivory Coast    rank 41–45


The index is robust to reweighting **distance, altitude and timezone** (τ ≈ 0.8–0.9)
but wobbles under *recovery_heavy* — group-stage rest days barely vary, so leaning
on them adds noise. That's a finding, not a flaw: the burden ranking is driven by
geography, and rest is a weak signal until the knockouts compress the calendar.

## 5 · Does travel actually hurt? A historical reality check

The index *measures* travel; it doesn't prove travel *matters*. So we test the
thesis on three past World Cups with real travel variance — South Africa 2010,
Brazil 2014, Russia 2018 — regressing each team's group-stage points on its
pre-tournament **Elo** (strength, built from the full match history) and its
within-tournament-standardized travel. Logic in `mlfootball/historical.py`.

In [6]:
from mlfootball import historical as H

reg = H.regression()
print(f"N = {reg['n']} team-campaigns · pooled R² = {reg['r2']}")
print(f"raw travel↔points correlation (uncontrolled): {reg['raw_travel_points_corr']}\n")
for name, c in reg["coef"].items():
    print(f"  {name:10} β={c['beta']:+.3f}  t={c['t']:+.2f}  p={c['p']:.3f}")

N = 96 team-campaigns · pooled R² = 0.3118


raw travel↔points correlation (uncontrolled): -0.1344

  intercept  β=+4.167  t=+19.16  p=0.000
  elo_z      β=+1.370  t=+6.30  p=0.000
  travel_z   β=-0.361  t=-1.66  p=0.097


**The verdict.** Strength (Elo) is overwhelmingly the story — but *holding strength
constant*, a standard deviation of extra group-stage travel costs about a third of
a point (β ≈ −0.36, p ≈ 0.10). Directionally real, modest, and noisy: travel is a
genuine headwind, not a death sentence. That honest "small but present" is exactly
how much weight the 2026 burden ranking deserves.

## 6 · Export for the web app

In [7]:
hist = {
    "n": reg["n"], "tournaments": reg["tournaments"], "r2": reg["r2"],
    "raw_corr": reg["raw_travel_points_corr"],
    "travel": reg["coef"]["travel_z"], "elo": reg["coef"]["elo_z"],
    "extremes": sorted(reg["panel"], key=lambda x: -x["travel_km"])[:5],
}
out = {
    "generated_note": "2026 WC group-stage travel burden. Static schedule + venue data.",
    "weights": T.DEFAULT_WEIGHTS,
    "alt_threshold": T.ALT_THRESHOLD,
    "venues": T.VENUES,
    "groups": T.GROUPS,
    "teams": table,
    "group_burden": T.group_burden(),
    "sensitivity": sens,
    "history": hist,
}
dest = Path(T.__file__).resolve().parent.parent / "site" / "data" / "travel.json"
dest.write_text(json.dumps(out, ensure_ascii=False))
print(f"wrote {dest} ({dest.stat().st_size/1024:.1f} KB)")

wrote /Users/williamcatt/Documents/Projects/worldcup2026/site/data/travel.json (46.7 KB)
